In [1]:
# ============================================================
# FDR correction for bilateral model-EEG spatial permutations
# ============================================================

import pandas as pd
from statsmodels.stats.multitest import multipletests
from google.colab import files

# ------------------------------------------------------------
# 1. Upload the two Excel files
# ------------------------------------------------------------
uploaded = files.upload()

visual_file = "permutaciones_espaciales_bilateral (1).xlsx"
auditory_file = "permutaciones_espaciales_bilateral.xlsx"

# ------------------------------------------------------------
# 2. Read the "resumen" sheet
# ------------------------------------------------------------
visual = pd.read_excel(visual_file, sheet_name="resumen")
auditory = pd.read_excel(auditory_file, sheet_name="resumen")

visual["modality"] = "Visual"
auditory["modality"] = "Auditory"

# ------------------------------------------------------------
# 3. Apply Benjamini-Hochberg FDR correction
#
# IMPORTANT:
# Correction is applied separately:
#   - for each modality
#   - for each correspondence metric
#   - across the three stimulation frequencies
# ------------------------------------------------------------

metrics = ["pearson", "spearman", "dice"]

for df in [visual, auditory]:

    for metric in metrics:

        p_col = f"{metric}_p_espacial"
        fdr_col = f"{metric}_p_FDR"

        p_values = df[p_col].values

        reject, p_fdr, _, _ = multipletests(
            p_values,
            alpha=0.05,
            method="fdr_bh"
        )

        df[fdr_col] = p_fdr
        df[f"{metric}_significant_FDR"] = reject


# ------------------------------------------------------------
# 4. Convert to long format for manuscript table
# ------------------------------------------------------------

rows = []

for df in [visual, auditory]:

    for _, row in df.iterrows():

        for metric in metrics:

            rows.append({
                "Modality": row["modality"],
                "Metric": metric.capitalize(),
                "Frequency_Hz": row["frecuencia_hz"],
                "Amplitude": row["amplitud"],
                "Observed_value": row[f"{metric}_observado"],
                "p_perm": row[f"{metric}_p_espacial"],
                "p_FDR": row[f"{metric}_p_FDR"],
                "Significant_FDR": row[f"{metric}_significant_FDR"]
            })

results = pd.DataFrame(rows)

# ------------------------------------------------------------
# 5. Sort the table
# ------------------------------------------------------------

metric_order = {
    "Pearson": 0,
    "Spearman": 1,
    "Dice": 2
}

results["metric_order"] = results["Metric"].map(metric_order)

results = (
    results
    .sort_values(["Modality", "metric_order", "Frequency_Hz"])
    .drop(columns="metric_order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

display(results)

Saving permutaciones_espaciales_bilateral (1).xlsx to permutaciones_espaciales_bilateral (1) (1).xlsx
Saving permutaciones_espaciales_bilateral.xlsx to permutaciones_espaciales_bilateral (2).xlsx


,Modality,Metric,Frequency_Hz,Amplitude,Observed_value,p_perm,p_FDR,Significant_FDR
0,Auditory,Pearson,13.000000,800,0.055132,0.320868,0.320868,False
1,Auditory,Pearson,29.400000,700,0.132220,0.120888,0.181332,False
2,Auditory,Pearson,43.000000,900,0.196910,0.034997,0.104990,False
3,Auditory,Spearman,13.000000,800,0.224269,0.013599,0.013599,True
4,Auditory,Spearman,29.400000,700,0.427674,0.000100,0.000150,True
5,Auditory,Spearman,43.000000,900,0.534734,0.000100,0.000150,True
6,Auditory,Dice,13.000000,800,0.360000,0.014499,0.014499,True
7,Auditory,Dice,29.400000,700,0.586207,0.000200,0.000300,True
8,Auditory,Dice,43.000000,900,0.724638,0.000100,0.000300,True
9,Visual,Pearson,13.000000,800,0.404154,0.000300,0.000900,True
